# Customer Churn Prediction

Predicting customer churn using the Telco Customer Churn dataset. Compares Logistic Regression, Random Forest, and XGBoost baselines, evaluated on precision/recall/F1 rather than accuracy alone due to class imbalance (~73/27 split).

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [2]:
df.shape



(7043, 21)

In [3]:

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Loading & Cleaning

`TotalCharges` loaded as an `object` dtype instead of numeric. Investigation showed 11 rows had blank values — all with `tenure = 0`, meaning these are new customers with no billing history yet. Imputed with 0 rather than median, since 0 is the mathematically correct value here, not a guess.

In [4]:
df.dtypes


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [5]:
df['TotalCharges'].str.strip().eq('').sum()

11

In [6]:
df[df['TotalCharges'].str.strip()=='']['tenure']

488     0
753     0
936     0
1082    0
1340    0
3331    0
3826    0
4380    0
5218    0
6670    0
6754    0
Name: tenure, dtype: int64

In [7]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [8]:
df['TotalCharges'].isnull().sum()


0

In [9]:
df.dtypes['TotalCharges']

dtype('float64')

## Encoding Categorical Features

Dropped `customerID` (unique identifier, no predictive value). Binary Yes/No columns mapped to 0/1. Columns like `OnlineSecurity` and `MultipleLines` had a third value (`"No internet/phone service"`) that was redundant with the `InternetService`/`PhoneService` columns — collapsed into `"No"` before encoding to reduce noise.

Remaining multi-category columns (`InternetService`, `Contract`, `PaymentMethod`) one-hot encoded with `drop_first=True` to avoid the dummy variable trap (multicollinearity from perfectly correlated dummy columns).


In [10]:
df = df.drop('customerID', axis=1)

In [11]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [12]:
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [13]:
df[['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,0,1,0,0,1,0
1,1,0,0,1,0,0
2,1,0,0,1,1,1
3,1,0,0,0,0,0
4,0,0,0,1,1,1


In [14]:
df['OnlineSecurity'].unique()

array(['No', 'Yes', 'No internet service'], dtype=object)

In [15]:
df['MultipleLines'].unique()

array(['No phone service', 'No', 'Yes'], dtype=object)

In [16]:
cols_with_service_dependency = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_with_service_dependency:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

In [17]:
for col in cols_with_service_dependency:
    print(col, df[col].unique())
    

MultipleLines ['No' 'Yes']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']


In [18]:
for col in cols_with_service_dependency:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [19]:
df[cols_with_service_dependency].head()

,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,0,0,1,0,0,0,0
1,0,1,0,1,0,0,0
2,0,1,1,0,0,0,0
3,0,1,0,1,1,0,0
4,0,0,0,0,0,0,0


In [20]:
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=True)

In [21]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,1,0,0,0,0,1,0


## Class Imbalance Check

Target is ~73% No Churn / 27% Churn. A model predicting "No churn" for everyone would score 73% accuracy while catching zero actual churners — this is why accuracy alone is a misleading metric here. The imbalance is moderate, not extreme, so no resampling (e.g. SMOTE) was applied — handled instead via metric choice (precision/recall/F1) at evaluation time.

In [22]:
df['Churn'].value_counts()

0    5174
1    1869
Name: Churn, dtype: int64

In [23]:
df['Churn'].value_counts(normalize = True)

0    0.73463
1    0.26537
Name: Churn, dtype: float64

## Train/Test Split

80/20 split, stratified on `Churn` to preserve the 73/27 ratio in both sets, with `random_state=42` for reproducibility.

In [24]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [25]:
y_train.value_counts(normalize=True)


0    0.734647
1    0.265353
Name: Churn, dtype: float64

In [26]:
y_test.value_counts(normalize=True)


0    0.734564
1    0.265436
Name: Churn, dtype: float64

## Feature Scaling

Scaled `tenure`, `MonthlyCharges`, `TotalCharges` using `StandardScaler`. Fit only on training data, then applied to test data — fitting on the full dataset (or test set) would leak test-set statistics into training, producing an overly optimistic evaluation.

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [28]:
X_train_scaled[numeric_cols].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-8.315835e-17,-2.410804e-16,-7.764073e-18
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


## Model 1: Logistic Regression (Baseline)

In [29]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

LogisticRegression(random_state=42)

In [30]:
y_pred = log_reg.predict(X_test_scaled)

In [31]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

[[927 108]
 [166 208]]


### Feature Coefficients

Strongest churn drivers: `InternetService_Fiber optic` (+), `TotalCharges` (+). Strongest retention factors: `Contract_Two year` (-), `tenure` (-), `InternetService_No` (-). Long-tenure, contract-locked customers rarely churn; fiber customers on flexible contracts are highest risk.

In [32]:
import pandas as pd

coefficients = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': log_reg.coef_[0]
})
coefficients = coefficients.sort_values('coefficient', ascending=False)
print(coefficients)

                                  feature  coefficient
16            InternetService_Fiber optic     1.049047
15                           TotalCharges     0.524088
21         PaymentMethod_Electronic check     0.384125
13                       PaperlessBilling     0.373840
6                           MultipleLines     0.335631
11                            StreamingTV     0.322066
12                        StreamingMovies     0.321683
1                           SeniorCitizen     0.146525
22             PaymentMethod_Mailed check     0.069836
0                                  gender     0.021666
2                                 Partner     0.021449
9                        DeviceProtection     0.009045
20  PaymentMethod_Credit card (automatic)    -0.033101
8                            OnlineBackup    -0.124253
3                              Dependents    -0.224951
14                         MonthlyCharges    -0.303074
10                            TechSupport    -0.324278
5         

## Model 2: Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

In [34]:
y_pred_rf = rf.predict(X_test_scaled)

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

[[927 108]
 [188 186]]


## Model 3: XGBoost

In [35]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_scaled, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [36]:
y_pred_xgb = xgb.predict(X_test_scaled)

print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.84      0.87      0.85      1035
           1       0.60      0.52      0.56       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.77      0.78      0.78      1409

[[903 132]
 [178 196]]


## Results Comparison

| Model | Precision (Churn) | Recall (Churn) | F1 (Churn) |
|---|---|---|---|
| Logistic Regression | 0.66 | 0.56 | **0.60** |
| Random Forest | 0.63 | 0.50 | 0.56 |
| XGBoost | 0.60 | 0.52 | 0.56 |

Logistic Regression outperformed both tree-based models on every metric for the churn class, despite being the simplest model. This suggests the churn signal is largely linear/additive — driven by contract length, tenure, and internet type — rather than requiring non-linear interactions. All models trained with default hyperparameters (no tuning); this is a baseline comparison, not a final performance ceiling.